# Notebook 46 — Four controls requested in review

**A. The 128-filter network at exactly 80% first-layer sparsity.** The width sweep (Notebook 45) bracketed 80% with 75% and 90%
cells; this runs the 80% cell (77 surviving weights) on the five 128-filter baselines.

**B. Dense versus sparse head under protected versus starved input.** A 2 x 2 factorial on the five 64-filter baselines with
conv.3 at 80% throughout: conv.0 protected or pruned to 80%, head dense or pruned to 80%. Two cells exist (Notebook 34);
the two dense-head cells are run here. If the starved-input, dense-head cell still collapses, the representation alone
carries the damage; if it recovers, the sparse head and its fine-tuning are part of the mechanism.

**C. Normalisation recalibration on the final fine-tuned checkpoints.** Batch-normalisation running statistics of the
uniformly pruned, fine-tuned models are recomputed on the training partition without changing any weight, and the
models are re-evaluated. The recalibration control in Section 4.3 was run before fine-tuning; this closes that gap.

**D. Blind-spot selection on validation records.** The threat model of Notebook 39b selected blind-spot classes with
attacker-side test-set recall and evaluated on the same test records. Here selection uses validation records and
evaluation uses untouched test records.

**Pre-stated criteria.** A: reported; the 80% cell should lie between the 75% and 90% cells. B: the sparse head is part of the
mechanism if the starved-input dense-head cell loses at least 0.10 less than the starved-input sparse-head cell; the
representation alone carries the damage if the dense-head cell still loses more than 0.15. C: normalisation is not the cause
if recalibration changes mean macro-F1 by less than 0.02. D: the recipe-conditional result stands if the uniform-over-protected
misattribution ratio on identical blind-spot traffic remains at least 2 with validation-based selection. GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
from itertools import combinations
ARCH = 'cnn1d'; KW64 = {'channels': (64, 128)}; KW128 = {'channels': (128, 128)}

print('controls A-D ready')

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Pruning policies on the CNN. Fine-tune loop identical to src.compression.prune_and_finetune.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def apply_layer_amounts(model, amounts):
    # amounts: {module_name: sparsity}; every prunable layer must be named (no silent defaults)
    m = copy.deepcopy(model); names = layer_names(m)
    for mod, name in prunable(m):
        a = amounts[names[mod]]
        if a > 0: prune.l1_unstructured(mod, name=name, amount=float(a)); prune.remove(mod, name)
    return m

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)


def load_cell(cell, seed, kw):
    m = M.build(ARCH, len(feat_cols), int(df.label.nunique()), **kw).to(DEVICE)
    m.load_state_dict(torch.load(PATHS.model(DATASET, ARCH, cell, seed), map_location=DEVICE, weights_only=False)['state_dict']); return m.eval()
def loss_table(name, rows, m0f1):
    d = pd.DataFrame(rows); g = d.groupby('cell').test_macro_f1.agg(['mean', 'std', 'min', 'max']); g['loss'] = m0f1 - g['mean']; print(f'\n{name}'); print(g.round(4).to_string()); return d, g
print('helpers ready')

In [ ]:
# ---------------- A: 128-filter network at exactly 80% first-layer sparsity (77 surviving weights) ----------------
rowsA, m0A = [], []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_f128_paired', seed, arch_kwargs=KW128)
    yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test'); m0A.append(f1_score(yt, pt, average='macro'))
    cell = 'f128_conv0dose800_paired'; p_c = PATHS.model(DATASET, ARCH, cell, seed)
    if os.path.exists(p_c): mp = load_cell(cell, seed, KW128); print(f'  loaded {cell} seed {seed}')
    else:
        mp, _, _ = finetune_masked(apply_layer_amounts(m0, {'conv.0': 0.8, 'conv.3': 0.8, 'head': 0.8}), seed); save_ckpt(mp, le, scaler, p_c); print(f'  saved {cell} seed {seed}')
    ls = layer_sparsity(mp); yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
    rowsA.append({'seed': seed, 'cell': cell, 'conv0_sparsity': ls['conv.0'], 'test_macro_f1': f1_score(yt, pc, average='macro')})
dA, gA = loss_table('A: 128 filters, conv.0 at 80%', rowsA, float(np.mean(m0A))); dA.to_csv(OUT / 'controls_A_f128_80_wide.csv', index=False)
ref = pd.read_csv(OUT / 'filter_sweep_dose_response.csv'); r128 = ref[ref.filters == 128].set_index('conv0_sparsity').mean_macro_f1_loss
lossA = float(gA.loc['f128_conv0dose800_paired', 'loss']); print(f'\n128-filter losses: 75% {r128.loc[0.75]:.3f} | 80% {lossA:.3f} | 90% {r128.loc[0.901]:.3f} | bracketed: {r128.loc[0.75] <= lossA <= r128.loc[0.901]}')

In [ ]:
# ---------------- B: dense vs sparse head under protected vs starved conv.0 (conv.3 at 80% throughout) ----------------
COND = {'starved_sparsehead': ('prune80_paired', {'conv.0': 0.8, 'conv.3': 0.8, 'head': 0.8}),
        'protected_sparsehead': ('layerwise80_protect_conv0_paired', {'conv.0': 0.0, 'conv.3': 0.8, 'head': 0.8}),
        'starved_densehead': ('ctrl_starved_densehead_paired', {'conv.0': 0.8, 'conv.3': 0.8, 'head': 0.0}),
        'protected_densehead': ('ctrl_protected_densehead_paired', {'conv.0': 0.0, 'conv.3': 0.8, 'head': 0.0})}
rowsB, m0B = [], []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=KW64)
    yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test'); m0B.append(f1_score(yt, pt, average='macro'))
    for name, (cell, amounts) in COND.items():
        p_c = PATHS.model(DATASET, ARCH, cell, seed)
        if os.path.exists(p_c): mp = load_cell(cell, seed, KW64); print(f'  loaded {name} seed {seed}')
        else:
            mp, _, _ = finetune_masked(apply_layer_amounts(m0, amounts), seed); save_ckpt(mp, le, scaler, p_c); print(f'  saved {name} seed {seed}')
        ls = layer_sparsity(mp); yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
        rowsB.append({'seed': seed, 'cell': name, 'conv0_sparsity': ls['conv.0'], 'head_sparsity': ls['head'], 'test_macro_f1': f1_score(yt, pc, average='macro')})
dB, gB = loss_table('B: 2x2 head x input factorial', rowsB, float(np.mean(m0B))); dB.to_csv(OUT / 'controls_B_head_factorial_wide.csv', index=False)
L = gB['loss']; dense_gain = float(L['starved_sparsehead'] - L['starved_densehead'])
print(f"\nstarved input: sparse head loss {L['starved_sparsehead']:.3f} vs dense head {L['starved_densehead']:.3f} (dense head recovers {dense_gain:.3f})")
print(f"protected input: sparse head {L['protected_sparsehead']:.3f} vs dense head {L['protected_densehead']:.3f}")
print('sparse head is part of the mechanism (dense head recovers >= 0.10):', dense_gain >= 0.10, '| representation alone collapses (dense-head cell loss > 0.15):', L['starved_densehead'] > 0.15)

In [ ]:
# ---------------- C: BatchNorm recalibration on the FINAL fine-tuned uniformly pruned checkpoints ----------------
from sklearn.preprocessing import LabelEncoder, StandardScaler
def recalibrate_bn(model, seed, n_batches=200):
    m = copy.deepcopy(model); le_ = LabelEncoder().fit(df['label'].to_numpy()); sc = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le_, sc); Xtr, _ = t['train']
    for mod in m.modules():
        if isinstance(mod, (nn.BatchNorm1d, nn.BatchNorm2d)): mod.reset_running_stats(); mod.momentum = None   # cumulative average
    m.train(); set_all_seeds(seed); idx = torch.randperm(len(Xtr))[: n_batches * 4096]
    with torch.no_grad():
        for i in range(0, len(idx), 4096): m(Xtr[idx[i:i + 4096]].to(DEVICE))
    return m.eval()
rowsC = []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=KW64)
    mp = load_cell('prune80_paired', seed, KW64); yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test'); before = f1_score(yt, pc, average='macro')
    mr = recalibrate_bn(mp, seed); assert layer_sparsity(mr)['prunable_sparsity'] == layer_sparsity(mp)['prunable_sparsity']
    yt, pr, _ = predict(mr, df, splits, le, scaler, feat_cols, which='test'); after = f1_score(yt, pr, average='macro')
    rowsC.append({'seed': seed, 'macro_f1_before': before, 'macro_f1_after_bn_recal': after, 'delta': after - before}); print(f'  seed {seed}: {before:.4f} -> {after:.4f}')
dC = pd.DataFrame(rowsC); dC.to_csv(OUT / 'controls_C_bn_recalibration_final.csv', index=False)
print(f"\nC: mean change from recalibration on final checkpoints: {dC.delta.mean():+.4f} (sd {dC.delta.std():.4f}) | normalisation not the cause (|mean change| < 0.02): {abs(dC.delta.mean()) < 0.02}")

In [ ]:
# ---------------- D: threat model with blind-spot selection on VALIDATION records, evaluation on TEST ----------------
fam_map = pd.read_csv(OUT / 'ciciot2023_alert_family_mapping.csv').set_index('fine_label')['alert_family'].to_dict()
RECIPES = {'default_layerwise80': 'prune80_paired', 'protect_conv0': 'layerwise80_protect_conv0_paired', 'global80': 'global80_paired'}
preds_val, preds_test = {}, {}
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=KW64)
    for recipe, model in [('dense', m0)] + [(r, load_cell(c, seed, KW64)) for r, c in RECIPES.items()]:
        yv, pv, _ = predict(model, df, splits, le, scaler, feat_cols, which='val'); yt, pt, _ = predict(model, df, splits, le, scaler, feat_cols, which='test')
        preds_val[(recipe, seed)] = (np.asarray(yv), np.asarray(pv)); preds_test[(recipe, seed)] = (np.asarray(yt), np.asarray(pt))
    print(f'  seed {seed}: predictions collected')
classes = list(le.classes_); C = len(classes); benign_idx = int(np.where(np.array(classes) == 'BenignTraffic')[0][0]); family_of = np.array([fam_map[c] for c in classes]); attack_idx = [i for i in range(C) if i != benign_idx]
def per_class_recall(yt, yp): return np.array([(yp[yt == c] == c).mean() if (yt == c).sum() else np.nan for c in range(C)])
recall_val = {k: per_class_recall(*v) for k, v in preds_val.items()}
def metrics(yt, yp, chosen):
    m = np.isin(yt, chosen); t, p = yt[m], yp[m]
    if len(t) == 0: return None
    to_benign = (p == benign_idx); cross = (family_of[p] != family_of[t]) & ~to_benign
    return {'misattribution_rate': float((p != t).mean()), 'attack_to_benign_rate': float(to_benign.mean()), 'cross_family_rate': float(cross.mean())}
rows, sets = [], []
for attacker_seeds in combinations(SEEDS, 3):
    held = [s for s in SEEDS if s not in attacker_seeds]
    r_dense = np.nanmean([recall_val[('dense', s)] for s in attacker_seeds], axis=0); r_pr = np.nanmean([recall_val[('default_layerwise80', s)] for s in attacker_seeds], axis=0)
    detectable = [c for c in attack_idx if r_dense[c] >= 0.5]; blind = [c for c in detectable if r_pr[c] < 0.2]
    sets.append({'attacker_seeds': str(attacker_seeds), 'n_blind': len(blind), 'blind_spots': ';'.join(classes[c] for c in blind)})
    for recipe in list(RECIPES) + ['dense']:
        for seed in held:
            yt, yp = preds_test[(recipe, seed)]
            for strategy, universe in (('random_over_detectable', detectable), ('blind_spot_set', blind)):
                per = [metrics(yt, yp, [c]) for c in universe]; per = [d for d in per if d]
                if not per: continue
                rows.append({'attacker_seeds': str(attacker_seeds), 'held_out_seed': seed, 'recipe': recipe, 'strategy': strategy, **{k: float(np.mean([d[k] for d in per])) for k in per[0]}})
fold = pd.DataFrame(rows); fold.to_csv(OUT / 'controls_D_threat_val_selection_fold_results.csv', index=False); pd.DataFrame(sets).to_csv(OUT / 'controls_D_threat_val_selection_blind_spots.csv', index=False)
summ = fold.groupby(['recipe', 'strategy'])[['misattribution_rate', 'attack_to_benign_rate', 'cross_family_rate']].mean().reset_index(); summ.to_csv(OUT / 'controls_D_threat_val_selection_summary.csv', index=False)
print(pd.DataFrame(sets).to_string(index=False)); print(); print(summ.round(4).to_string(index=False))
def g_(r): return float(summ[(summ.recipe == r) & (summ.strategy == 'blind_spot_set')].misattribution_rate.iloc[0])
ratio = g_('default_layerwise80') / g_('protect_conv0'); print(f"\nD: uniform-over-protected misattribution ratio on identical blind-spot traffic (validation-selected): {ratio:.3f} | stands (>= 2): {ratio >= 2}")

In [ ]:
# ---------------- Verdict file + commit ----------------
verdict = pd.DataFrame([
 {'control': 'A_f128_at_80pct', 'value': f'loss {lossA:.3f} (75%: {r128.loc[0.75]:.3f}, 90%: {r128.loc[0.901]:.3f})', 'pass': bool(r128.loc[0.75] <= lossA <= r128.loc[0.901])},
 {'control': 'B_sparse_head_part_of_mechanism_dense_recovers_ge_0.10', 'value': round(dense_gain, 4), 'pass': bool(dense_gain >= 0.10)},
 {'control': 'B_representation_alone_collapses_dense_head_loss_gt_0.15', 'value': round(float(L['starved_densehead']), 4), 'pass': bool(L['starved_densehead'] > 0.15)},
 {'control': 'C_bn_recal_final_abs_mean_change_lt_0.02', 'value': round(float(dC.delta.mean()), 4), 'pass': bool(abs(dC.delta.mean()) < 0.02)},
 {'control': 'D_val_selected_ratio_ge_2', 'value': round(ratio, 3), 'pass': bool(ratio >= 2)},
]); print(verdict.to_string(index=False)); verdict.to_csv(OUT / 'controls_gate_verdict.csv', index=False)
write_json(OUT / 'controls_environment.json', {'seeds': SEEDS, 'environment': environment_record()})
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip(); assert _b == 'main', f'checked-out branch is {_b!r}'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True); subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred): shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/46_review_controls.ipynb'
if os.path.exists(_own):
    d_ = _json.load(open(_own))
    for c in d_.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d_, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/controls_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 46: review controls - 128-filter 80% cell, dense-vs-sparse head factorial, BatchNorm recalibration on final checkpoints, validation-selected blind spots'], capture_output=True, text=True); print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed'); print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)